# Exploration des ratios financiers (BCE / INPI)

Ici on regarde le fichier des **ratios financiers** : des indicateurs calculés à partir des comptes annuels que les entreprises déposent au greffe. C'est ce qui va nous permettre de décrire la santé financière des entreprises avant une éventuelle faillite.

⚠️ **RAM** : comme pour SIRENE, on charge tout via `scripts/data_utils.py` (`load_ratios()` lit un parquet préparé à partir du CSV). Après une étape lourde : `del` puis `free_memory()`.

### Lexique : les comptes et les ratios

Chaque année, beaucoup d'entreprises déposent leurs **comptes annuels** au greffe :
- le **bilan** : ce que l'entreprise possède et ce qu'elle doit, à la date de clôture ;
- le **compte de résultat** : ce qu'elle a gagné et dépensé sur l'année.

L'année comptable s'appelle un **exercice**. Elle se termine à la **date de clôture** (souvent le 31 décembre, mais pas toujours). ⚠️ Rien à voir avec la fermeture de l'entreprise : il y a une clôture chaque année.

Un **ratio**, c'est une division entre deux chiffres de ces comptes. Ça permet de comparer des entreprises de tailles différentes : 1 M€ de dettes, c'est énorme pour une petite boîte et rien pour un grand groupe, mais des dettes égales à 3 fois les fonds propres, c'est inquiétant dans les deux cas.

| Colonne | En clair |
|---|---|
| `chiffre_d_affaires` | Ce que l'entreprise a vendu sur l'année (en €). |
| `marge_brute` | Les ventes moins le coût de ce qui a été acheté pour être revendu ou transformé (en €). |
| `ebe` | Excédent brut d'exploitation : ce que l'activité rapporte avant l'usure du matériel, les intérêts et les impôts. En gros, est-ce que le métier de base gagne de l'argent ? (en €) |
| `ebit` | Résultat d'exploitation : l'EBE moins l'usure du matériel (amortissements) (en €). |
| `resultat_net` | Le bénéfice (ou la perte) final, après tout (en €). |
| `marge_ebe` | EBE / chiffre d'affaires : la part des ventes qui reste après les dépenses courantes. |
| `caf_sur_ca` | Capacité d'autofinancement / chiffre d'affaires : l'argent que l'entreprise génère elle-même, rapporté à ses ventes. |
| `resultat_courant_avant_impots_sur_ca` | Le résultat avant impôts et éléments exceptionnels, rapporté aux ventes. |
| `taux_d_endettement` | Dettes financières / capitaux propres. Plus c'est haut, plus l'entreprise dépend de ses prêteurs. |
| `autonomie_financiere` | Capitaux propres / total du bilan : la part de l'entreprise financée par ses propres fonds. |
| `capacite_de_remboursement` | Dettes financières / CAF : le nombre d'années qu'il faudrait pour rembourser les dettes avec l'argent généré. |
| `couverture_des_interets` | Est-ce que les bénéfices suffisent à payer les intérêts des emprunts ? |
| `ratio_de_liquidite` | Ce qui peut vite devenir du cash / les dettes à court terme. En dessous de 100 %, l'entreprise risque de ne pas pouvoir payer ses dettes du moment. |
| `ratio_de_vetuste` | À quel point le matériel est usé (amortissements / valeur d'achat). |
| `poids_bfr_exploitation_sur_ca` (et `_jours`) | Le besoin en fonds de roulement : l'argent bloqué dans les stocks et les factures pas encore payées par les clients, rapporté aux ventes (ou en nombre de jours de ventes). |
| `rotation_des_stocks_jours` | Combien de jours le stock reste en moyenne avant d'être vendu. |
| `credit_clients_jours` | En combien de jours les clients paient, en moyenne. |
| `credit_fournisseurs_jours` | En combien de jours l'entreprise paie ses fournisseurs, en moyenne. |
| `type_bilan` | `C` = bilan complet, `S` = bilan simplifié (petites entreprises), `K` = bilan consolidé (les comptes d'un groupe entier, maison mère + filiales). |
| `confidentiality` | Si l'entreprise a demandé à garder une partie de ses comptes confidentielle. |

Ces définitions viennent de la comptabilité générale, pas d'une doc officielle de la source : à vérifier avant de les citer.

## 1. Chargement

In [1]:
import sys
sys.path.append("../scripts")

import pandas as pd
import numpy as np
from data_utils import load_ratios, load_sirene_ul, free_memory, mem

pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

r = load_ratios()
print(r.shape, f"{r.memory_usage(deep=True).sum() / 1e9:.2f} Go |", mem())

(6813145, 23) 0.64 Go | process : 1.37 Go | disponible : 9.74 Go


In [2]:
r.head()

,siren,date_cloture_exercice,chiffre_d_affaires,marge_brute,ebe,ebit,resultat_net,taux_d_endettement,ratio_de_liquidite,ratio_de_vetuste,autonomie_financiere,poids_bfr_exploitation_sur_ca,couverture_des_interets,caf_sur_ca,capacite_de_remboursement,marge_ebe,resultat_courant_avant_impots_sur_ca,poids_bfr_exploitation_sur_ca_jours,rotation_des_stocks_jours,credit_clients_jours,credit_fournisseurs_jours,type_bilan,confidentiality
0,844516393,2023-12-31,0.00,0.00,"-8,233.00","-8,233.00","752,563.00",0.25,191.17,NaN,0.24,NaN,0.00,NaN,0.00,NaN,NaN,NaN,NaN,NaN,452.86,S,Public
1,844517946,2024-12-31,"492,542.00","492,542.00","-1,048,336.00","-1,059,966.00","18,937,724.00",0.00,700.33,97.49,97.67,-84.83,-9.54,-210.56,0.00,-212.84,-148.49,-305.40,0.00,0.00,4.71,C,Public
2,844518340,2020-12-31,0.00,0.00,0.00,0.00,"52,453.00",3.64,117.67,67.81,18.72,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C,Partiellement confidentiel
3,844518951,2020-09-30,"2,288,570.00","1,609,290.00","47,457.00","22,162.00","15,416.00",437.99,163.74,84.52,7.55,57.22,3.87,0.85,26.83,2.07,0.97,206.00,17.76,81.24,104.22,C,Public
4,844519330,2019-12-31,0.00,0.00,"-2,439.00","-2,439.00","-2,448.00",0.21,0.04,NaN,99.35,NaN,-0.37,NaN,-0.45,NaN,NaN,NaN,NaN,NaN,287.82,C,Public


In [3]:
r.dtypes

siren                                            int64
date_cloture_exercice                   datetime64[us]
chiffre_d_affaires                             float32
marge_brute                                    float32
ebe                                            float32
ebit                                           float32
resultat_net                                   float32
taux_d_endettement                             float32
ratio_de_liquidite                             float32
ratio_de_vetuste                               float32
autonomie_financiere                           float32
poids_bfr_exploitation_sur_ca                  float32
couverture_des_interets                        float32
caf_sur_ca                                     float32
capacite_de_remboursement                      float32
marge_ebe                                      float32
resultat_courant_avant_impots_sur_ca           float32
poids_bfr_exploitation_sur_ca_jours            float32
rotation_d

### Observations

- Le fichier contient **6 813 145 lignes** pour **1 615 938 entreprises**. En RAM, ça prend environ 0,6 Go.
- Les montants (chiffre d'affaires, EBE…) sont en euros, les ratios sont des nombres à virgule, et on a deux colonnes texte (`type_bilan`, `confidentiality`).

## 2. Une ligne, c'est quoi ?

In [4]:
print("lignes :", len(r))
print("entreprises (SIREN) :", r["siren"].nunique())
print("lignes identiques en double :", r.duplicated().sum())
cle = ["siren", "date_cloture_exercice"]
doublons = r.duplicated(cle, keep=False)
print("lignes avec le même SIREN et la même date de clôture :", doublons.sum())

lignes : 6813145


entreprises (SIREN) : 1615938


lignes identiques en double : 0


lignes avec le même SIREN et la même date de clôture : 64029


In [5]:
# Les doublons SIREN + date : quel type de bilan ?
pd.crosstab(r.loc[doublons, "type_bilan"], r.loc[doublons, "confidentiality"], margins=True)

confidentiality,Public,Partiellement confidentiel,Partiellement confidentiel (RAPCAC),Publication simplifiee,All
type_bilan,,,,,
S,2399,332,5,3,2739
C,30856,445,103,8,31412
K,29851,22,5,0,29878
All,63106,799,113,11,64029


In [6]:
# Exemple : une entreprise avec un bilan complet (C) et un bilan consolidé (K) la même année
exemple = r.loc[doublons, "siren"].iloc[0]
r.loc[r["siren"] == exemple, cle + ["type_bilan", "chiffre_d_affaires", "resultat_net"]].sort_values(cle)

,siren,date_cloture_exercice,type_bilan,chiffre_d_affaires,resultat_net
917199,844549147,2023-12-31,C,0.00,"-13,654,544.00"
3009790,844549147,2023-12-31,K,"614,252,992.00","-7,010,000.00"
34,844549147,2024-12-31,K,"579,310,016.00","-17,460,000.00"
6490669,844549147,2024-12-31,C,"1,050,000.00","-12,294,393.00"


In [7]:
# Si on enlève les bilans consolidés (K), reste-t-il des doublons ?
sans_k = r[r["type_bilan"] != "K"]
d2 = sans_k.duplicated(cle, keep=False)
print("doublons hors K :", d2.sum())
display(pd.crosstab(sans_k.loc[d2, "type_bilan"], sans_k.loc[d2, "confidentiality"]))
sans_k.loc[d2].sort_values(cle).head(6)[cle + ["type_bilan", "confidentiality", "chiffre_d_affaires", "resultat_net"]]

doublons hors K : 4294


confidentiality,Public,Partiellement confidentiel,Partiellement confidentiel (RAPCAC),Publication simplifiee
type_bilan,,,,
S,1810,331,3,3
C,1796,345,4,2


,siren,date_cloture_exercice,type_bilan,confidentiality,chiffre_d_affaires,resultat_net
2563648,66803479,2019-12-31,S,Public,"6,919,000.00","710,000.00"
3195522,66803479,2019-12-31,C,Public,"6,926,000.00","712,000.00"
1363189,86380359,2018-03-31,C,Public,"836,649.00","107,070.00"
5404251,86380359,2018-03-31,S,Public,"509,060.00","77,136.00"
992048,86380359,2019-03-31,C,Public,"11,495.00","-2,295.00"
6596815,86380359,2019-03-31,S,Public,"509,060.00","77,136.00"


In [8]:
del sans_k, d2
free_memory()

'process : 1.00 Go | disponible : 10.07 Go'

### Observations

- Il n'y a **aucune ligne copiée en double**, mais **64 029 lignes partagent le même SIREN et la même date de clôture** qu'une autre.
- La plupart du temps, c'est parce qu'un **groupe** a déposé deux bilans la même année : le bilan de la société seule (`C`) et le bilan **consolidé** du groupe entier (`K`). Dans l'exemple, la même société affiche 0 € de chiffre d'affaires dans son bilan `C` en 2023, mais 614 M€ dans le bilan `K` du groupe. Ce ne sont donc pas les mêmes chiffres du tout.
- Si on enlève les `K`, il reste quand même **4 294 lignes en doublon**, avec un bilan simplifié (`S`) **et** un bilan complet (`C`) pour la même entreprise et la même date. Parfois les chiffres sont presque pareils (6 919 000 € contre 6 926 000 €), parfois pas du tout. On voit même un bilan `S` qui a exactement les mêmes chiffres deux années de suite (509 060 € en 2018 et en 2019), ce qui ressemble à une erreur de la source.

## 3. Les dates de clôture

In [9]:
annee = r["date_cloture_exercice"].dt.year
annee.value_counts().sort_index()

date_cloture_exercice
1919         1
2002         1
2003         1
2004         2
2005         1
2006         2
2007         3
2008         1
2009         7
2010        26
2011        47
2012      2831
2013     12582
2014     30033
2015    118859
2016    672987
2017    722960
2018    701918
2019    704147
2020    693717
2021    687268
2022    679321
2023    707763
2024    666054
2025    405531
2026      7081
2029         1
Name: count, dtype: int64

In [10]:
# Mois de clôture
r["date_cloture_exercice"].dt.month.value_counts().sort_index()

date_cloture_exercice
1       55676
2       37932
3      428823
4       62134
5       36479
6      505537
7       71442
8      152829
9      708837
10      91675
11      39769
12    4622012
Name: count, dtype: int64

In [11]:
# Nombre d'exercices par entreprise (hors bilans consolidés)
nb_exercices = r[r["type_bilan"] != "K"].groupby("siren").size()
display(nb_exercices.describe())
nb_exercices.value_counts().sort_index()

count   1,615,694.00
mean            4.20
std             2.99
min             1.00
25%             2.00
50%             3.00
75%             7.00
max            15.00
dtype: float64

1     397125
2     254302
3     190770
4     145660
5     117455
6     100095
7      91243
8      94308
9     120708
10     94645
11      7596
12      1341
13       387
14        55
15         4
Name: count, dtype: int64

In [12]:
# Écart entre deux clôtures successives d'une même entreprise (hors K), en mois
s = r.loc[r["type_bilan"] != "K", cle].drop_duplicates().sort_values(cle)
ecart = (s.groupby("siren")["date_cloture_exercice"].diff().dt.days / 30.44).round()
print("écarts calculés :", ecart.notna().sum())
display(ecart.describe())
ecart.clip(upper=36).value_counts().sort_index().loc[lambda v: v > 1000]

écarts calculés : 5162179


count   5,162,179.00
mean           13.61
std             6.84
min             0.00
25%            12.00
50%            12.00
75%            12.00
max         1,176.00
Name: date_cloture_exercice, dtype: float64

date_cloture_exercice
3.00        2202
4.00        1097
6.00        6074
7.00        1302
8.00        2127
9.00        7719
10.00       2253
11.00       2237
12.00    4694114
13.00       2823
14.00       3067
15.00       9176
16.00       2157
17.00       1094
18.00       6393
21.00       2177
24.00     262987
27.00       1376
30.00       1124
36.00     143921
Name: count, dtype: int64

In [13]:
del s, ecart, nb_exercices
free_memory()

'process : 1.03 Go | disponible : 9.92 Go'

### Observations

- Les exercices vont surtout de **2016 à 2024** (environ 700 000 par an). Avant 2016, il n'y a presque rien (118 859 en 2015, encore moins avant) : le fichier commence vraiment en 2016. **2025 n'est pas complète** (405 531) et on a quelques dates bizarres (1919, 2029).
- La plupart des entreprises clôturent au **31 décembre** (4,6 M de lignes sur 6,8 M). Les autres mois fréquents sont septembre, juin et mars.
- En moyenne, une entreprise a **4,2 exercices** dans le fichier (entre 1 et 15). 397 125 entreprises n'en ont qu'un seul.
- Entre deux clôtures d'une même entreprise, il y a presque toujours **12 mois** (4,7 M de cas sur 5,2 M). Mais on a aussi **262 987 trous de 24 mois** et **143 921 de 36 mois** : il manque une ou deux années de comptes au milieu. Soit l'entreprise ne les a pas déposés, soit ils ne sont pas dans le fichier.

## 4. Type de bilan et confidentialité

In [14]:
pd.crosstab(r['type_bilan'], r['confidentiality'], margins=True)

confidentiality,Public,Partiellement confidentiel,Partiellement confidentiel (RAPCAC),Publication simplifiee,All
type_bilan,,,,,
S,1433819,202188,728,480,1637215
C,3674429,1409857,56310,2209,5142805
K,33076,42,7,0,33125
All,5141324,1612087,57045,2689,6813145


In [15]:
# Part de valeurs à 0 ou vides selon la confidentialité, pour quelques colonnes
cols_montants = ["chiffre_d_affaires", "marge_brute", "ebe", "ebit", "resultat_net"]
cols_ratios = ["taux_d_endettement", "ratio_de_liquidite", "autonomie_financiere", "marge_ebe", "credit_clients_jours"]
(r.groupby("confidentiality", observed=True)[cols_montants + cols_ratios]
   .agg(lambda s: (s.isna() | (s == 0)).mean()).round(3).T)

confidentiality,Public,Partiellement confidentiel,Partiellement confidentiel (RAPCAC),Publication simplifiee
chiffre_d_affaires,0.16,0.95,1.00,0.04
marge_brute,0.16,0.95,1.00,0.04
ebe,0.01,0.94,1.00,0.01
ebit,0.01,0.94,1.00,0.00
resultat_net,0.01,0.12,0.02,0.00
taux_d_endettement,0.15,0.06,0.07,0.11
ratio_de_liquidite,0.04,0.02,0.02,0.04
autonomie_financiere,0.06,0.02,0.00,0.05
marge_ebe,0.16,0.95,1.00,0.04
credit_clients_jours,0.36,0.87,0.95,0.14


In [16]:
# Chez les comptes publics : chiffre d'affaires à 0 selon le type de bilan
public = r["confidentiality"] == "Public"
pd.crosstab(r.loc[public, "type_bilan"], r.loc[public, "chiffre_d_affaires"] == 0, colnames=["CA = 0"], normalize="index").round(3)

CA = 0,False,True
type_bilan,,
S,0.85,0.15
C,0.83,0.17
K,0.94,0.06


### Observations

- **75 % des lignes sont publiques** (5 141 324). 1 612 087 sont **partiellement confidentielles**, 57 045 en « RAPCAC » et 2 689 en « publication simplifiée ».
- ⚠️ **Gros piège : quand les comptes sont confidentiels, les montants ne sont pas vides mais à 0.** Chez les « partiellement confidentiel », le chiffre d'affaires, la marge brute, l'EBE et la marge d'EBE valent 0 dans 95 % des cas, et dans 100 % des cas chez les RAPCAC. Par contre, le résultat net et les ratios d'endettement ou de liquidité sont souvent quand même remplis.
- Même chez les comptes **publics**, le chiffre d'affaires vaut 0 dans **16 %** des cas. Ça peut être un vrai 0 (par exemple une société qui ne vend rien, comme une holding ou une SCI), mais on ne peut pas le savoir juste avec ce fichier.

## 5. Valeurs manquantes

In [17]:
# Part de cases vides (NaN) par colonne, selon le type de bilan
r.groupby("type_bilan", observed=True).apply(lambda g: g.isna().mean()).T.round(3)

type_bilan,S,C,K
siren,0.00,0.00,0.00
date_cloture_exercice,0.00,0.00,0.00
chiffre_d_affaires,0.00,0.00,0.00
marge_brute,0.00,0.00,0.00
ebe,0.00,0.00,0.00
ebit,0.00,0.00,0.00
resultat_net,0.00,0.00,0.00
taux_d_endettement,0.00,0.00,0.03
ratio_de_liquidite,0.02,0.01,0.04
ratio_de_vetuste,0.24,0.19,0.71


### Observations

- Les **montants** (chiffre d'affaires, marge brute, EBE, EBIT, résultat net) ne sont **jamais vides**. Comme vu juste avant, ils sont à 0 quand on ne les connaît pas.
- Les **ratios** sont plus souvent vides : 24 à 39 % pour la marge d'EBE, la CAF, le BFR, les stocks et le crédit clients, et jusqu'à 71 % pour la vétusté dans les bilans consolidés.
- Le taux d'endettement, l'autonomie financière et la liquidité sont presque toujours remplis (moins de 5 % de vides).

## 6. Dans quelle unité sont les ratios ?

In [18]:
# Si marge_ebe = EBE / CA x 100, alors le rapport ci-dessous vaut 100
ok = (r["chiffre_d_affaires"] > 0) & r["marge_ebe"].notna()
(r.loc[ok, "marge_ebe"] / (r.loc[ok, "ebe"] / r.loc[ok, "chiffre_d_affaires"])).describe()

count   4,385,596.00
mean           98.98
std             6.47
min        -1,698.88
25%            99.99
50%           100.00
75%           100.00
max         3,211.11
dtype: float64

In [19]:
# Même idée pour caf_sur_ca et resultat_courant_avant_impots_sur_ca : on regarde juste leur ordre de grandeur
r.loc[ok, ["chiffre_d_affaires", "ebe", "marge_ebe", "caf_sur_ca", "resultat_courant_avant_impots_sur_ca"]].head(8)

,chiffre_d_affaires,ebe,marge_ebe,caf_sur_ca,resultat_courant_avant_impots_sur_ca
1,"492,542.00","-1,048,336.00",-212.84,-210.56,-148.49
3,"2,288,570.00","47,457.00",2.07,0.85,0.97
5,"42,522.00","2,836.00",6.67,5.46,4.66
6,"3,417,417.00","27,674.00",0.80,0.63,0.78
7,"4,725,892.00","190,757.00",4.03,3.92,0.16
8,"146,292.00","40,199.00",27.11,23.49,20.14
10,"172,755.00","10,191.00",5.90,6.03,5.07
11,"159,396.00","37,614.00",23.60,13.37,7.58


### Observations

- Quand on recalcule EBE / chiffre d'affaires, on retombe sur la `marge_ebe` **multipliée par 100** (médiane du rapport : 100,00). Donc **les ratios sont en pourcentage** : une marge d'EBE de 6,18 veut dire 6,18 %.
- Les autres ratios en « sur CA » (CAF, résultat courant) ont le même ordre de grandeur, donc ils ont l'air d'être aussi en %.

## 7. Distribution des valeurs

In [20]:
num = r.select_dtypes("number").columns.drop("siren")
quantiles = r[num].quantile([0, 0.01, 0.25, 0.5, 0.75, 0.99, 1]).T
quantiles.columns = ["min", "1 %", "25 %", "médiane", "75 %", "99 %", "max"]
quantiles

,min,1 %,25 %,médiane,75 %,99 %,max
chiffre_d_affaires,"-1,073,450,944.00",0.00,0.00,"76,990.00","681,389.00","75,923,432.00","293,544,774,336,512.00"
marge_brute,"-442,765,661,962,240.00",0.00,0.00,"59,654.00","460,357.00","41,436,808.00","197,542,709,035,008.00"
ebe,"-442,765,896,843,264.00","-845,129.25",-917.00,0.00,"41,581.00","6,221,670.50","312,373,306,982,400.00"
ebit,"-181,892,368,302,080.00","-971,087.00","-1,899.00",0.00,"29,565.00","3,789,044.50","983,047,012,876,288.00"
resultat_net,"-55,779,438,592.00","-1,482,388.25",-522.00,"15,804.00","89,240.00","4,572,099.00","4,202,842,292,224.00"
taux_d_endettement,"-787,512,064.00","-2,659.84",0.02,16.89,86.82,"4,513.24","701,832,512.00"
ratio_de_liquidite,"-814,370,624.00",0.00,113.28,198.19,395.13,"31,220.19","2,207,073,792.00"
ratio_de_vetuste,"-4,609,600.00",0.00,16.95,35.83,61.08,100.00,"97,663.42"
autonomie_financiere,"-1,498,800,128.00",-371.83,9.93,36.44,63.56,125.08,"3,262,170,624.00"
poids_bfr_exploitation_sur_ca,"-70,566,469,632.00","-1,271.21",-5.76,9.42,30.35,"2,694.90","19,441,924,096.00"


In [21]:
# Part de valeurs négatives et de valeurs à 0 (comptes publics seulement, pour ne pas compter les zéros de confidentialité)
pd.DataFrame({
    "négatif": (r.loc[public, num] < 0).mean(),
    "zéro": (r.loc[public, num] == 0).mean(),
}).round(3)

,négatif,zéro
chiffre_d_affaires,0.00,0.16
marge_brute,0.01,0.16
ebe,0.36,0.01
ebit,0.40,0.01
resultat_net,0.31,0.01
taux_d_endettement,0.13,0.14
ratio_de_liquidite,0.00,0.02
ratio_de_vetuste,0.00,0.05
autonomie_financiere,0.10,0.06
poids_bfr_exploitation_sur_ca,0.28,0.00


In [22]:
# Les plus gros chiffres d'affaires du fichier
r.nlargest(10, "chiffre_d_affaires")[cle + ["type_bilan", "chiffre_d_affaires", "resultat_net"]]

,siren,date_cloture_exercice,type_bilan,chiffre_d_affaires,resultat_net
2591476,955200928,2016-12-31,K,"293,544,774,336,512.00","2,778,347,077,632.00"
4079453,348607417,2017-12-31,K,"197,542,709,035,008.00","148,320,000.00"
2745624,428760847,2020-12-31,K,"855,646,994,432.00",0.00
4976518,842442972,2020-12-31,K,"707,128,000,512.00",0.00
460228,352256424,2023-12-31,K,"628,220,559,360.00","1,398,109,952.00"
4197759,342376332,2025-12-31,C,"268,352,552,960.00","1,421,755,416,576.00"
1740091,404495566,2017-12-31,K,"266,287,300,608.00","9,873,829,888.00"
4001581,452847130,2025-03-31,K,"244,987,396,096.00","4,149,849,088.00"
3695547,542051180,2023-12-31,K,"237,128,007,680.00","21,510,000,640.00"
1411324,452847130,2024-03-31,K,"230,023,102,464.00","3,587,065,088.00"


In [23]:
# Les ratios de liquidité les plus élevés
r.nlargest(5, "ratio_de_liquidite")[cle + ["type_bilan", "chiffre_d_affaires", "ratio_de_liquidite"]]

,siren,date_cloture_exercice,type_bilan,chiffre_d_affaires,ratio_de_liquidite
368840,529174609,2014-12-31,C,0.00,"2,207,073,792.00"
3131085,429483902,2017-09-30,C,0.00,"1,712,239,488.00"
1913841,423123033,2024-12-31,C,0.00,"1,283,710,080.00"
909627,834740771,2020-12-31,C,0.00,"1,135,134,976.00"
5537451,752455998,2025-12-31,C,0.00,"824,348,416.00"


### Observations

- Les médianes ont l'air normales : chiffre d'affaires de 76 990 €, résultat net de 15 804 €, taux d'endettement de 16,9 %, liquidité de 198 %, autonomie financière de 36 %, clients payés en 30 jours, fournisseurs payés en 49 jours.
- Par contre, les **extrêmes sont complètement fous** : un chiffre d'affaires de 2,9 × 10¹⁴ € (c'est plus de 100 fois le PIB de la France), une marge brute minimale de −4,4 × 10¹⁴ €, un ratio de liquidité de 2,2 milliards de %. Les plus gros chiffres d'affaires sont surtout des bilans consolidés (`K`), et les liquidités énormes viennent d'entreprises avec un chiffre d'affaires à 0.
- On a aussi des valeurs **impossibles** : des chiffres d'affaires négatifs (jusqu'à −1 milliard €), des délais clients ou fournisseurs négatifs, des taux de vétusté négatifs alors qu'un taux d'usure devrait être entre 0 et 100 %.
- Chez les comptes publics, il y a beaucoup de **pertes** : l'EBE est négatif dans 36 % des cas, et le résultat net dans 31 %.

## 8. Qui sont ces entreprises ? (lien avec SIRENE)

In [24]:
ul = load_sirene_ul(columns=["siren", "categorieJuridiqueUniteLegale", "activitePrincipaleUniteLegale",
                             "nomenclatureActivitePrincipaleUniteLegale", "etatAdministratifUniteLegale"])
ul = ul[ul["siren"].isin(r["siren"].unique())]
free_memory()
print("entreprises des ratios trouvées dans SIRENE :", len(ul), "sur", r["siren"].nunique())

entreprises des ratios trouvées dans SIRENE : 1615890 sur 1615938


In [25]:
cj = ul["categorieJuridiqueUniteLegale"].astype("string")
display(cj.str[0].value_counts())
cj.value_counts().head(10)

categorieJuridiqueUniteLegale
5    1589336
6      16192
1       8343
3       1908
9         48
4         43
2         19
8          1
Name: count, dtype: int64[pyarrow]

categorieJuridiqueUniteLegale
5710    770085
5499    729052
5485     29924
5202     19587
5599     13016
1000      8343
6540      8182
5785      6981
5770      6153
5470      5698
Name: count, dtype: int64[pyarrow]

In [26]:
# Section NAF (la lettre) à partir de la division (2 chiffres), pour les codes NAFRev2
SECTIONS = {"A": (1, 3), "B": (5, 9), "C": (10, 33), "D": (35, 35), "E": (36, 39), "F": (41, 43),
            "G": (45, 47), "H": (49, 53), "I": (55, 56), "J": (58, 63), "K": (64, 66), "L": (68, 68),
            "M": (69, 75), "N": (77, 82), "O": (84, 84), "P": (85, 85), "Q": (86, 88), "R": (90, 93),
            "S": (94, 96), "T": (97, 98), "U": (99, 99)}
NOMS = {"A": "Agriculture", "B": "Industries extractives", "C": "Industrie manufacturière", "D": "Énergie",
        "E": "Eau, déchets", "F": "Construction", "G": "Commerce", "H": "Transports", "I": "Hébergement, restauration",
        "J": "Information, communication", "K": "Finance, assurance", "L": "Immobilier",
        "M": "Activités spécialisées, scientifiques", "N": "Services administratifs", "O": "Administration publique",
        "P": "Enseignement", "Q": "Santé, action sociale", "R": "Arts, spectacles", "S": "Autres services",
        "T": "Ménages employeurs", "U": "Extraterritorial"}

def section(code_naf):
    div = pd.to_numeric(code_naf.astype("string").str[:2], errors="coerce")
    out = pd.Series(pd.NA, index=code_naf.index, dtype="string")
    for lettre, (a, b) in SECTIONS.items():
        out[div.between(a, b)] = lettre
    return out

rev2 = ul["nomenclatureActivitePrincipaleUniteLegale"] == "NAFRev2"
sec = section(ul.loc[rev2, "activitePrincipaleUniteLegale"])
sec.map(NOMS).value_counts()

Commerce                                 307975
Activités spécialisées, scientifiques    219531
Construction                             199707
Finance, assurance                       186750
Hébergement, restauration                126513
Immobilier                               114785
Industrie manufacturière                  95432
Services administratifs                   89791
Information, communication                66794
Transports                                54438
Autres services                           41814
Santé, action sociale                     28294
Enseignement                              21328
Énergie                                   19245
Arts, spectacles                          17629
Agriculture                               16501
Eau, déchets                               5386
Industries extractives                     1454
Administration publique                      46
Ménages employeurs                            1
Name: count, dtype: int64

In [27]:
ul['etatAdministratifUniteLegale'].value_counts()

etatAdministratifUniteLegale
A    1255204
C     360686
Name: count, dtype: int64

### Observations

- On retrouve dans SIRENE **1 615 890 des 1 615 938 entreprises** du fichier (48 absentes).
- Ce sont presque toutes des **sociétés commerciales** (famille `5` : 98 %), surtout des SAS (`5710` : 770 085) et des SARL (`5499` : 729 052). Il n'y a que 8 343 entrepreneurs individuels.
- Les secteurs les plus présents : le commerce (307 975), les activités spécialisées et scientifiques (219 531), la construction (199 707) et la finance-assurance (186 750, sûrement beaucoup de holdings).
- **1 255 204 sont encore actives** et 360 686 sont fermées.

## 9. Les entreprises du BODACC ont-elles des ratios ?

On prend les entreprises citées au BODACC en 2019 et la **première date de parution** de chacune en 2019. On regarde si elles ont des ratios, et combien de temps sépare leur **dernière clôture** de cette première parution. Ici on utilise la date de parution (plus simple à récupérer) et pas la date du jugement, qui est un peu plus tôt.

In [28]:
bod = pd.read_csv("../data/raw/bodacc/procedures_collectives_2019.csv.gz", sep=";", dtype=str,
                  usecols=["dateparution", "registre"])
bod["siren"] = bod["registre"].str.findall(r"\b\d{9}\b")
bod = bod.explode("siren").dropna(subset=["siren"])
bod["siren"] = bod["siren"].astype("int64")
premiere = bod.groupby("siren")["dateparution"].min().pipe(pd.to_datetime)
del bod
avec = premiere.index.isin(r["siren"].unique())
print(f"entreprises BODACC 2019 : {len(premiere)} | avec au moins un exercice dans les ratios : {avec.sum()} ({avec.mean():.1%})")

entreprises BODACC 2019 : 121579 | avec au moins un exercice dans les ratios : 21629 (17.8%)


In [29]:
# Dernière clôture AVANT la première parution (hors bilans consolidés)
rr = r.loc[(r["type_bilan"] != "K") & r["siren"].isin(premiere.index), cle].copy()
rr["parution"] = rr["siren"].map(premiere)
avant = rr[rr["date_cloture_exercice"] < rr["parution"]]
derniere = avant.groupby("siren")["date_cloture_exercice"].max()
ecart_mois = ((premiere.loc[derniere.index] - derniere).dt.days / 30.44)
print("entreprises avec une clôture avant leur parution :", len(derniere))
print("entreprises dont tous les exercices sont après la parution :", rr["siren"].nunique() - len(derniere))
display(ecart_mois.describe())
pd.cut(ecart_mois, [0, 6, 12, 18, 24, 36, 1000], labels=["< 6 mois", "6-12 mois", "12-18 mois", "18-24 mois", "24-36 mois", "> 36 mois"]).value_counts().sort_index()

entreprises avec une clôture avant leur parution : 21035
entreprises dont tous les exercices sont après la parution : 592


count   21,035.00
mean        22.77
std         12.15
min          0.03
25%         13.44
50%         23.16
75%         30.85
max         84.86
dtype: float64

< 6 mois      1867
6-12 mois     2636
12-18 mois    3371
18-24 mois    3083
24-36 mois    7468
> 36 mois     2610
Name: count, dtype: int64

### Observations

- Sur les 121 579 entreprises citées au BODACC en 2019, seulement **21 629 (17,8 %) ont au moins un exercice** dans le fichier des ratios.
- 21 035 ont une clôture **avant** leur première parution. Pour 592 autres, tous les exercices du fichier sont **après** la parution (l'entreprise a continué à déposer des comptes, par exemple pendant un redressement).
- Entre la **dernière clôture** et la première parution, il se passe en moyenne **23 mois** (médiane 23,2 mois). Seulement 1 867 entreprises ont des comptes de moins de 6 mois avant la parution, et 2 610 ont des comptes vieux de plus de 3 ans.
- ⚠️ Pour 2019, une partie de l'écart vient du fait que le fichier commence vraiment en 2016 : une entreprise ne peut pas avoir de comptes plus anciens. Dans les faits, les derniers comptes qu'on a avant une parution ont souvent entre un et deux ans.

In [30]:
del rr, avant, derniere, ecart_mois, premiere, ul
print(free_memory())

process : 1.18 Go | disponible : 9.85 Go


## Récapitulatif des décisions pour le nettoyage

| # | Sujet | Décision |
|---|---|---|
| 1 | **Bilan de la société seule ou du groupe (`K`)** | On garde le bilan de la **société seule** (`C` ou `S`). On garde le bilan du **groupe** (`K`) seulement quand il n'y a pas de bilan de la société seule pour la même date. |
| 2 | **Doublons bilan simplifié + bilan complet** | On n'en garde **qu'un** : le **bilan complet** (`C`), le plus détaillé. |
| 3 | **Montants à 0** | Comptes **publics** : les 0 sont **gardés à 0 €**, on part du principe que la donnée est correcte. Comptes **confidentiels** (partiellement confidentiel, RAPCAC) : les 0 ne sont **pas gardés**, ils passent en valeur manquante pour ne pas fausser les données (ce sont des montants cachés, pas de vrais 0). |
| 4 | **Valeurs extrêmes** | **Winsorisation** colonne par colonne : tout ce qui est sous la valeur du 1 % le plus bas, ou au-dessus de celle du 1 % le plus haut, est ramené à cette valeur. Les seuils sont calculés **après** les autres étapes de nettoyage. Pour chaque colonne winsorisée, on ajoute une colonne oui/non (`<colonne>_ramene`) qui dit si la valeur a été ramenée. La winsorisation règle aussi les valeurs impossibles : pour le chiffre d'affaires, la vétusté et les délais, la limite basse vaut 0, donc les valeurs négatives sont ramenées à 0. |
| 5 | **Années gardées** | Les exercices clôturés **de 2016 à aujourd'hui**. Ça enlève les quelques exercices plus anciens et les dates dans le futur (2029). |
| 6 | **Comment on relie comptes et faillites** | **Une ligne = un exercice** (une entreprise, une année de comptes), pour **toutes** les entreprises, en faillite ou non. Chaque exercice reçoit une étiquette : **jugement d'ouverture dans les 36 mois après la date de clôture, oui ou non**. Pour la date, on prend le **jugement** et pas la parution. Les comptes clôturés plus de 36 mois avant le jugement sont étiquetés « non ». Les comptes clôturés **après fin octobre 2023** sont mis de côté : leurs 36 mois ne sont pas encore écoulés (les données s'arrêtent fin octobre 2026), donc on ne peut pas dire « non » avec certitude. |
| 7 | **Déséquilibre entre les deux groupes** | Il y a beaucoup moins d'exercices « faillite » que « sans faillite ». On en tiendra compte au moment de l'analyse (méthode à choisir à ce moment-là). |
| 8 | **Taille de l'entreprise** | On la mesure avec le **chiffre d'affaires après winsorisation**. Les très grandes entreprises sont donc toutes ramenées à la même limite haute (environ 77,6 M€), on l'accepte : la winsorisation sert à corriger les valeurs absurdes. |
| 9 | **Exercices d'entreprises déjà passées par une défaillance** | **Gardés** dans l'analyse : l'entreprise peut avoir connu une autre décision de justice depuis (plan de redressement par exemple). Repérables avec la colonne `deja_defaillante`. |
